# 01 — Arrays and Two Pointers

## Why This Matters for DSA
This opens Phase 02 because two pointers is the single highest-leverage pattern for array problems: it routinely turns an obvious O(n²) brute force into O(n), using zero extra memory, once you recognize the shape of the problem. It also directly sets up `02_Sliding_Window_and_Prefix_Sum` (same-direction pointers with a moving window) and the cycle-detection you'll reuse in `07_Linked_Lists`. If you only internalize one pattern from this entire phase, make it this one — it appears constantly, in disguise, across nearly every later topic.

## Prerequisites
- `03_STL_Deep_Dive` (Phase 00) — `vector` operations, iterators
- `01_Complexity_Analysis` (Phase 01) — recognizing when a change in algorithm shape changes the complexity class

## Learning Objectives
By the end of this notebook, you will be able to:
- Recognize when a problem's structure allows opposite-direction two pointers to replace an O(n²) brute force with O(n)
- Apply the read/write same-direction pointer pattern for in-place array modification
- Use Floyd's cycle detection (fast/slow pointers) on an array treated as an implicit linked structure
- Merge two sorted sequences with two pointers, the same core step used inside merge sort
- Partition an array into three regions in a single pass (Dutch National Flag)
- Judge whether a problem's structure actually supports two pointers, or whether it's a trap that looks similar but isn't

## Checklist Before Moving On
- [ ] I can explain why Two Sum works with two pointers on a SORTED array but not on an unsorted one
- [ ] I can state the greedy justification for why Container With Most Water always moves the shorter pointer
- [ ] I know the difference between a read/write pointer pair and an opposite-direction pointer pair, and when to use each
- [ ] I can trace Floyd's cycle detection by hand on a small example and explain why the two-phase approach finds the cycle's entrance specifically
- [ ] I can write the merge step of merge sort from memory
- [ ] I can explain, for a new problem, whether its structure actually supports two pointers before reaching for the pattern out of habit

## Table of Contents
1. Two Pointers — Opposite Direction
2. Container With Most Water — The Greedy Two-Pointer Movement Rule
3. Two Pointers — Same Direction (Read/Write Pointers)
4. Fast and Slow Pointers — Floyd's Cycle Detection
5. Merging Two Sorted Arrays
6. Three Pointers — Dutch National Flag Partitioning
7. When Two Pointers Works (and When It Doesn't)
8. Phase Checkpoint, Cheat Sheet, and Answer Key
9. LeetCode Practice Problems


## 1. Two Pointers — Opposite Direction

### The Why
This is the pattern that turns "check every pair" (O(n²)) into "check every element once" (O(n)) — but only when the array's structure (usually sortedness) lets you make a provably correct decision about which pointer to move, instead of trying every combination blindly.

### Core Mechanism
- One pointer (`left`) starts at index 0, the other (`right`) starts at the last index. They move toward each other until they meet or cross.
- **In-place reversal:** swap the elements at `left` and `right`, then move both inward — O(n) time, O(1) space, versus allocating a whole new reversed array.
- **Two Sum on a sorted array:** because the array is sorted, the current pair's sum tells you EXACTLY which direction to move. Sum too low? The only way to increase it is a bigger left value, so move `left` up. Sum too high? Move `right` down. This decision is only valid BECAUSE the array is sorted — an unsorted array gives you no such guarantee.
- **Why this beats brute force:** checking every pair is O(n²) and re-examines pairs redundantly. Two pointers never revisits a pair — each pointer only ever moves toward the other, so the total number of pointer moves is bounded by O(n).

### Common Pitfalls
- Applying this pattern to an UNSORTED array expecting it to still work — without sortedness (or some other monotonic structure), there's no valid basis for deciding which pointer to move, and the algorithm silently produces wrong answers.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
using namespace std;

// TWO POINTERS, OPPOSITE DIRECTION: one pointer starts at the front, one at the back,
// and they move TOWARD each other. This pattern works whenever a decision at the two
// ends can be made independent of everything in the middle.

void reverseInPlace(vector<int>& v) {
    int left = 0, right = (int)v.size() - 1;
    while (left < right) {
        swap(v[left], v[right]);
        left++;
        right--;
    }
    // O(n) time, O(1) extra space -- compare to allocating a new reversed array, which
    // would be O(n) space for no benefit here
}

// classic use case: Two Sum on a SORTED array. Because the array is sorted, you can
// reason about which pointer to move: if the current pair sums too LOW, the only way to
// increase the sum is to move `left` up (to a bigger value); if too HIGH, move `right` down.
vector<int> twoSumSorted(vector<int>& nums, int target) {
    int left = 0, right = (int)nums.size() - 1;
    while (left < right) {
        int sum = nums[left] + nums[right];
        if (sum == target) return {left, right};
        else if (sum < target) left++;    // need a bigger sum -- move left pointer up
        else right--;                      // need a smaller sum -- move right pointer down
    }
    return {-1, -1};
}

int main() {
    vector<int> v{1, 2, 3, 4, 5};
    reverseInPlace(v);
    for (int x : v) cout << x << " ";
    cout << "\n";

    vector<int> sorted{2, 7, 11, 15};
    auto result = twoSumSorted(sorted, 9);
    cout << "indices: " << result[0] << ", " << result[1] << "\n";

    // WHY THIS IS BETTER THAN BRUTE FORCE: checking every pair is O(n^2). Because the
    // array is sorted, two pointers narrows the search space by ONE element on every
    // step (never revisiting a pair), giving O(n) -- the sortedness is what MAKES this
    // pointer-movement decision valid; this pattern requires sorted (or otherwise
    // monotonic) structure to work correctly

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 2. Container With Most Water — The Greedy Two-Pointer Movement Rule

### The Why
This problem is the standard test of whether you actually understand WHY two pointers works, rather than just pattern-matching "two pointers" to "array problem." The movement rule here needs its own proof — it's not simply "smaller side moves" by convention, it's the ONLY choice that can possibly improve the answer.

### Core Mechanism
- Area between two lines = `width × min(height[left], height[right])` — the shorter line always caps the height, no matter how tall the other one is.
- **The greedy rule:** always move the pointer at the SHORTER line. Reasoning: if you moved the TALLER line's pointer instead, the width shrinks, and the height is still capped by the same (unmoved) shorter line — so the new area is guaranteed to be no better, often worse. Moving the shorter line's pointer is the only move that has any chance of finding a taller line that could improve the capped height, even though width shrinks either way.
- This greedy justification — "the alternative move is PROVABLY never better" — is what makes it safe to never reconsider a discarded pointer position. That's what makes the algorithm O(n) instead of needing to explore multiple movement choices.

### Common Pitfalls
- Moving the taller pointer "just to try it" or alternating arbitrarily — without the greedy proof, you'd need to explore multiple movement choices per step, which reintroduces the O(n²) cost this pattern is supposed to eliminate.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <algorithm>
using namespace std;

// Container With Most Water: given heights, find two lines that, together with the
// x-axis, form a container holding the most water. Area = width * min(height[left], height[right]).
int maxArea(vector<int>& height) {
    int left = 0, right = (int)height.size() - 1;
    int best = 0;

    while (left < right) {
        int width = right - left;
        int h = min(height[left], height[right]);
        best = max(best, width * h);

        // KEY GREEDY INSIGHT: always move the pointer at the SHORTER line inward.
        // Moving the taller line's pointer can only ever DECREASE the width while the
        // height is still capped by the (unchanged) shorter line -- guaranteed worse.
        // Moving the shorter line's pointer is the only move that has any CHANCE of
        // finding a taller line that improves the area, even though width shrinks either way.
        if (height[left] < height[right]) left++;
        else right--;
    }
    return best;
}

int main() {
    vector<int> height{1, 8, 6, 2, 5, 4, 8, 3, 7};
    cout << "max area: " << maxArea(height) << "\n";   // expected 49 (lines at index 1 and 8)

    // this is O(n) with two pointers, versus O(n^2) checking every pair of lines --
    // the greedy pointer-movement rule is what makes it safe to never "go back"

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 3. Two Pointers — Same Direction (Read/Write Pointers)

### The Why
This is the go-to pattern for in-place array modification — filtering, deduplicating, compacting — without allocating a second array. Both pointers move forward, but at different effective speeds, and the gap between them is exactly the answer you're building.

### Core Mechanism
- The **write pointer** tracks where the next "kept" element should go. The **read pointer** scans ahead, one element at a time, deciding what's worth keeping.
- **Remove duplicates from a sorted array:** since duplicates are always adjacent in a sorted array, compare each read element only against the last WRITTEN element — if different, it's genuinely new, write it and advance `write`. If the same, skip it — `read` moves on, `write` stays put.
- **Move zeroes:** swap the read element into the write position whenever it's non-zero, advancing `write` only then. Zero elements get left behind as `write` and `read` diverge, naturally ending up pushed to the back.
- In both cases, `write` never moves faster than `read`, which is what guarantees no element gets overwritten before it's been examined — this invariant is worth checking explicitly whenever you write a read/write pointer solution.

### Common Pitfalls
- Advancing the write pointer unconditionally (every iteration) instead of only when a value is actually kept — this defeats the entire purpose and just copies the array as-is.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
using namespace std;

// TWO POINTERS, SAME DIRECTION (read/write pointers): both pointers move forward, but
// at different speeds. One pointer (write/slow) tracks "where the next kept element goes,"
// the other (read/fast) scans ahead looking for elements worth keeping.

// remove duplicates from a SORTED array in place, return the new logical length
int removeDuplicates(vector<int>& nums) {
    if (nums.empty()) return 0;
    int write = 1;   // first element is always kept -- nothing to compare it against yet
    for (int read = 1; read < (int)nums.size(); read++) {
        if (nums[read] != nums[write - 1]) {   // found a genuinely new value
            nums[write] = nums[read];
            write++;
        }
        // if nums[read] == nums[write-1], it's a duplicate -- read moves on, write stays put
    }
    return write;   // everything from index 0 to write-1 is the deduplicated result
}

// move all zeroes to the end, keeping the relative order of non-zero elements, in place
void moveZeroes(vector<int>& nums) {
    int write = 0;   // tracks where the next non-zero element should go
    for (int read = 0; read < (int)nums.size(); read++) {
        if (nums[read] != 0) {
            swap(nums[write], nums[read]);
            write++;
        }
    }
    // after the loop, everything before `write` is non-zero (in original relative order),
    // and everything from `write` onward is zero -- the swaps push zeroes rightward naturally
}

int main() {
    vector<int> dups{1, 1, 2, 2, 2, 3, 4, 4};
    int newLen = removeDuplicates(dups);
    cout << "deduplicated: ";
    for (int i = 0; i < newLen; i++) cout << dups[i] << " ";
    cout << "(new length=" << newLen << ")\n";

    vector<int> zeros{0, 1, 0, 3, 12};
    moveZeroes(zeros);
    cout << "after moveZeroes: ";
    for (int x : zeros) cout << x << " ";
    cout << "\n";

    // both are O(n) time, O(1) extra space -- the write pointer never moves faster than
    // the read pointer, so no element is ever overwritten before it's been read

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 4. Fast and Slow Pointers — Floyd's Cycle Detection

### The Why
This is the standard tool for detecting a cycle without extra memory, and it's not limited to linked lists (where you'll meet it again in `07_Linked_Lists`) — it applies to ANY sequence where each position deterministically points to a "next" position, including an array where values are treated as indices.

### Core Mechanism
- Two pointers traverse the same sequence at different speeds: `slow` advances one step at a time, `fast` advances two. If there's a cycle, `fast` will eventually "lap" `slow` from behind and they'll meet somewhere INSIDE the cycle — if there's no cycle, `fast` simply reaches the end first.
- **The array-as-implicit-linked-list trick (Find the Duplicate Number):** given `n+1` integers all in range `[1, n]`, treat `nums[i]` as "an edge from index `i` to index `nums[i]`." A repeated value means two different indices point to the same next index — which creates a cycle, and the duplicate value is exactly that cycle's entrance.
- **Two-phase algorithm:** Phase 1 runs slow/fast until they meet anywhere inside the cycle (this only proves a cycle exists and finds A meeting point, not necessarily the entrance). Phase 2 resets one pointer to the start and moves BOTH pointers one step at a time — they're mathematically guaranteed to meet exactly at the cycle's entrance, a consequence of the distances involved being equal (the proof itself is a nice bit of algebra, worth working through on paper once, though not required to use the technique).
- **Why this beats the obvious approach:** a hash set to track "seen" values also finds the duplicate in O(n) time, but costs O(n) extra space. Floyd's approach gets the same O(n) time with O(1) extra space — pure pointer movement, no auxiliary structure at all.

### Common Pitfalls
- Stopping after Phase 1 and returning the first meeting point as "the answer" — that meeting point is guaranteed to be SOMEWHERE in the cycle, but not necessarily the cycle's entrance (the actual duplicate value). Phase 2 is not optional.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
using namespace std;

// FAST AND SLOW POINTERS (Floyd's cycle detection, "tortoise and hare"): two pointers
// traverse the same sequence at different speeds -- slow moves 1 step, fast moves 2 steps.
// If there's a cycle, fast will eventually lap slow and they'll meet INSIDE the cycle.
// This pattern is the standard tool for detecting cycles in linked lists (next notebook),
// but it also applies to an array treated as an implicit linked list via VALUE-AS-INDEX --
// which is exactly the trick behind "Find the Duplicate Number."

// nums has n+1 integers, each in [1, n] -- by pigeonhole, at least one value repeats.
// Treat nums[i] as "a pointer from index i to index nums[i]" -- since a duplicate value
// means two different indices point to the SAME next index, this creates a cycle, and
// the duplicate value is exactly the entry point of that cycle.
int findDuplicate(vector<int>& nums) {
    int slow = nums[0], fast = nums[0];

    // Phase 1: detect that a cycle exists, and find A meeting point (not necessarily the start)
    do {
        slow = nums[slow];
        fast = nums[nums[fast]];
    } while (slow != fast);

    // Phase 2: find the cycle's ENTRANCE. Reset one pointer to the start, move both at
    // speed 1 -- they're mathematically guaranteed to meet exactly at the cycle's entrance.
    // (This is a property of Floyd's algorithm: the distance from the start to the cycle
    // entrance equals the distance from the meeting point to the cycle entrance, going
    // around the cycle.)
    slow = nums[0];
    while (slow != fast) {
        slow = nums[slow];
        fast = nums[fast];
    }
    return slow;
}

int main() {
    vector<int> nums{1, 3, 4, 2, 2};
    cout << "duplicate: " << findDuplicate(nums) << "\n";   // expected 2

    vector<int> nums2{3, 1, 3, 4, 2};
    cout << "duplicate: " << findDuplicate(nums2) << "\n";  // expected 3

    // O(n) time, O(1) extra space -- no hash set needed, purely pointer movement.
    // Compare to the "obvious" hash-set approach: O(n) time but O(n) EXTRA space --
    // Floyd's trick gets the same time complexity with no extra memory at all

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 5. Merging Two Sorted Arrays

### The Why
This is the merge step of merge sort, isolated and studied on its own — because "merge two sorted things" reappears constantly outside of sorting too (merging intervals, merging sorted linked lists, k-way merges). Understanding it cleanly here makes every later appearance a five-second recognition instead of a fresh derivation.

### Core Mechanism
- One pointer per array, both starting at index 0. At each step, compare the two pointers' current values and append the SMALLER one to the result, advancing only that pointer.
- Once one array is exhausted, the other's remaining elements are already sorted relative to each other and all bigger than everything placed so far — just append them directly, no more comparisons needed.
- Each pointer only ever moves forward, and every element from both arrays is examined exactly once — that's what makes this O(n + m) rather than anything involving re-comparison or re-scanning.

### Common Pitfalls
- Forgetting the two "drain the leftovers" loops at the end — if you stop as soon as EITHER pointer reaches its array's end, you'll silently drop the remaining elements of whichever array had more left.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
using namespace std;

// TWO POINTERS ACROSS TWO DIFFERENT ARRAYS: one pointer per array, both moving forward,
// always advancing whichever pointer currently points at the smaller value. This is the
// merge step of merge sort -- worth knowing on its own, since "merge two sorted things"
// shows up constantly outside of sorting too (merging intervals, merging sorted lists).
vector<int> mergeSorted(vector<int>& a, vector<int>& b) {
    vector<int> result;
    result.reserve(a.size() + b.size());
    int i = 0, j = 0;

    while (i < (int)a.size() && j < (int)b.size()) {
        if (a[i] <= b[j]) {
            result.push_back(a[i]);
            i++;
        } else {
            result.push_back(b[j]);
            j++;
        }
    }
    // exactly one of these two loops actually runs -- whichever array still has leftovers
    while (i < (int)a.size()) result.push_back(a[i++]);
    while (j < (int)b.size()) result.push_back(b[j++]);

    return result;
}

int main() {
    vector<int> a{1, 3, 5, 7};
    vector<int> b{2, 4, 6, 8, 10};
    auto merged = mergeSorted(a, b);

    cout << "merged: ";
    for (int x : merged) cout << x << " ";
    cout << "\n";

    // O(n + m) time -- each element from both arrays is visited exactly once, and each
    // pointer only ever moves forward, never backward or re-checks an element

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 6. Three Pointers — Dutch National Flag Partitioning

### The Why
Two pointers generalizes cleanly to three when a problem has three distinct categories instead of two — and this specific problem (sorting an array of only 0s, 1s, and 2s) is the classic demonstration, named after the three horizontal bands of the Dutch flag.

### Core Mechanism
- Three pointers (`low`, `mid`, `high`) divide the array into four conceptual regions as `mid` scans through: confirmed 0s before `low`, confirmed 1s between `low` and `mid`, unprocessed unknowns between `mid` and `high`, and confirmed 2s after `high`.
- `nums[mid] == 0`: swap it into the confirmed-0s region (swap with `low`), advance BOTH `low` and `mid` — the value swapped in at `mid` is safe to treat as processed, since it came from the confirmed-1s region.
- `nums[mid] == 1`: already correctly placed, just advance `mid`.
- `nums[mid] == 2`: swap it to the confirmed-2s region (swap with `high`), advance ONLY `high` — deliberately do NOT advance `mid`, because the value just swapped in from `high` is completely unknown and must still be examined.
- The result is a full sort in a SINGLE pass, O(n) time and O(1) space — versus a general sort at O(n log n) for information you already have (only 3 possible values), the same idea behind counting sort exploiting a limited value range.

### Common Pitfalls
- Advancing `mid` after the 2-swap case — this is the one asymmetry in the algorithm, and skipping it (treating all three cases identically) silently leaves unexamined values in the array, producing an incorrect partition.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
using namespace std;

// THREE POINTERS (Dutch National Flag partitioning): sort an array of only 0s, 1s, and 2s
// in a SINGLE pass, in place. Three pointers divide the array into four regions as they move:
// [0, low)      -- confirmed 0s
// [low, mid)    -- confirmed 1s
// [mid, high]   -- unprocessed, unknown values
// (high, end)   -- confirmed 2s
void sortColors(vector<int>& nums) {
    int low = 0, mid = 0, high = (int)nums.size() - 1;

    while (mid <= high) {
        if (nums[mid] == 0) {
            swap(nums[low], nums[mid]);
            low++;
            mid++;   // the swapped-in value at mid is now confirmed processed (it came from
                     // the "confirmed 1s" region or was itself already checked), safe to advance
        } else if (nums[mid] == 1) {
            mid++;   // already in the correct region, just move on
        } else {   // nums[mid] == 2
            swap(nums[mid], nums[high]);
            high--;
            // deliberately do NOT advance mid here -- the value swapped in from `high` is
            // UNKNOWN and hasn't been checked yet, so mid must re-examine this same position
        }
    }
}

int main() {
    vector<int> nums{2, 0, 2, 1, 1, 0};
    sortColors(nums);
    for (int x : nums) cout << x << " ";
    cout << "\n";   // expected: 0 0 1 1 2 2

    // O(n) single pass, O(1) space -- versus sorting normally which would be O(n log n)
    // for information you already have (only 3 distinct values) -- this pattern exploits
    // that limited-value-range structure the same way counting sort does

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 7. When Two Pointers Works (and When It Doesn't)

### The Why
Two pointers is easy to reach for out of habit once it feels familiar — but it only works when the problem's structure genuinely supports a provably correct pointer-movement rule. Recognizing the difference is what separates confident pattern application from cargo-culting a technique that happens to have worked on the last three problems.

### Core Mechanism
- **Two pointers works when:** the array is sorted (or otherwise monotonic), AND you can prove — the way Container With Most Water's greedy rule was proven — that moving one specific pointer is never worse than the alternative. If you can't articulate WHY a particular pointer should move, that's a signal the justification might not actually exist for this problem.
- **Two pointers does NOT work when:** the array is unsorted and sorting it would destroy information you need (e.g. original indices matter and can't be recovered by tracking them separately), or the problem doesn't reduce to a monotonic decision at two ends — many problems that "involve two things moving through an array" are actually sliding window (`02_Sliding_Window_and_Prefix_Sum`) or a different pattern entirely, not opposite/same-direction two pointers.
- **A quick self-check:** can you state, in one sentence, the RULE for which pointer moves and why it's always correct to make that move? If yes, you likely have a valid two-pointer solution. If you're moving pointers by trial and error or backtracking pointer positions, you don't actually have a two-pointer algorithm — you have a search dressed up to look like one, and its real complexity is probably worse than O(n).

### Common Pitfalls
- Sorting an array to enable two pointers when the problem needs the ORIGINAL indices in the answer — sorting destroys index information; if you need it, sort a list of (value, original index) pairs instead, or track indices separately.


## 8. Phase Checkpoint, Cheat Sheet, and Answer Key

### Two Pointers Pattern Cheat Sheet

| Variant | Pointer movement | Requires | Complexity | Example |
|---|---|---|---|---|
| Opposite direction | Both move toward each other | Sorted / monotonic structure | O(n) | Two Sum II, reverse array |
| Greedy opposite direction | Move the pointer that can't possibly be optimal to keep | A provable "never worse" argument | O(n) | Container With Most Water |
| Same direction (read/write) | Both move forward, write ≤ read | In-place filtering/compaction | O(n) | Remove Duplicates, Move Zeroes |
| Fast/slow | Different speeds (1 step vs 2 steps) | A deterministic "next" relationship (cycle possible) | O(n) time, O(1) space | Find the Duplicate Number, cycle detection |
| Merge (two arrays) | One pointer per array, both forward | Both inputs already sorted | O(n+m) | Merge Sorted Array, merge step of merge sort |
| Three pointers | Three regions, asymmetric advance rules | A small fixed number of categories | O(n) | Sort Colors (Dutch National Flag) |

### Checkpoint Questions
1. Why does opposite-direction two pointers require the array to be sorted (or otherwise monotonic)?
2. In Container With Most Water, why is it always safe to move the pointer at the shorter line, and never the taller one?
3. What invariant must hold between the read and write pointers in the same-direction pattern, and what goes wrong if it's violated?
4. In Floyd's cycle detection, why isn't the first meeting point (end of Phase 1) necessarily the cycle's entrance?
5. When merging two sorted arrays, why are the two "drain the leftovers" loops at the end both necessary, even though only one of them will actually execute any iterations on a given run?
6. In Dutch National Flag partitioning, why does the `mid` pointer advance after swapping a 0, but NOT after swapping a 2?

### Answer Key
1. Because the decision of which pointer to move relies on knowing that moving `left` forward only ever increases values, and moving `right` backward only ever decreases them. Without sortedness, you have no guarantee about what value you'll encounter next, so there's no valid basis for choosing a direction.
2. Because the container's height is always capped by the SHORTER of the two lines. Moving the taller line's pointer can only shrink the width while the height stays capped by the same shorter line — guaranteed no better. Moving the shorter line's pointer is the only move that could possibly find a taller line and improve the capped height.
3. The write pointer must never move ahead of the read pointer (`write <= read` at all times). If violated, you'd write to a position whose original value hasn't been read yet, silently corrupting data you still needed to examine.
4. Floyd's Phase 1 only proves a cycle exists and finds SOME point within it — the mathematical property that guarantees meeting exactly at the entrance requires the specific Phase 2 procedure (reset one pointer to the start, advance both at equal speed) built on the equal-distance relationship between the start, the entrance, and the meeting point.
5. Because you don't know in advance which array will be exhausted first — it depends on the actual data. Whichever array still has elements left after the main loop ends needs its remainder appended directly (already sorted, all bigger than what's been placed) — omitting either loop risks silently dropping elements on inputs where that particular array happens to be the one with leftovers.
6. Swapping in a 0 pulls from the confirmed-1s region (between `low` and `mid`), whose value is already known-safe to treat as processed — so `mid` can safely advance. Swapping in a 2 pulls from the completely UNPROCESSED region (beyond `high`), so the newly swapped-in value at `mid` hasn't been examined yet and must be checked again before advancing past it.

### Mini Project
Implement `threeSum(nums)`: find all unique triplets in an array that sum to zero.
1. Sort the array first (this problem needs sortedness to work).
2. Fix one element with an outer loop, then use OPPOSITE-DIRECTION two pointers on the remaining sub-array to find pairs that sum to the negation of the fixed element (exactly the Two Sum II pattern from Section 1, reapplied inside a loop).
3. Handle duplicate triplets: after finding a valid triplet, skip over any adjacent duplicate values for both the outer loop variable and the two pointers, so the same triplet of VALUES isn't recorded more than once.

This exercises: the opposite-direction pattern from Section 1, layered inside another loop, plus the practical bookkeeping of duplicate-avoidance that real two-pointer problems often need beyond the "clean" examples in this notebook.


## 9. LeetCode Practice Problems

Work these in order within each group — easy problems first to lock in the pattern's mechanics, then medium/hard to test whether you can adapt it. Problem numbers are LeetCode's; difficulty is LeetCode's official rating. Hints point at the pattern, not the solution — resist looking past the hint until you've genuinely tried.

### Opposite-Direction Two Pointers

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 125 | Valid Palindrome | Easy | Skip non-alphanumeric characters as you converge `left`/`right` inward |
| 344 | Reverse String | Easy | Direct application of Section 1's `reverseInPlace` |
| 167 | Two Sum II - Input Array Is Sorted | Easy | Exactly Section 1's `twoSumSorted` |
| 977 | Squares of a Sorted Array | Easy | The largest square is always at one of the two ends — think about why |
| 15 | 3Sum | Medium | This notebook's Mini Project — fix one element, two-pointer the rest |
| 18 | 4Sum | Medium | Extend 3Sum's idea one level deeper (two fixed elements, then two pointers) |
| 42 | Trapping Rain Water | Hard | Related to Container With Most Water, but the answer is a SUM across all positions, not a single max — think about what bounds the water level at each index |

### Same-Direction (Read/Write) Two Pointers

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 26 | Remove Duplicates from Sorted Array | Easy | Direct application of Section 3 |
| 27 | Remove Element | Easy | Same read/write pattern, condition is "not equal to a target value" instead of "not a duplicate" |
| 283 | Move Zeroes | Easy | Direct application of Section 3 |
| 80 | Remove Duplicates from Sorted Array II | Medium | Same idea as #26, but allow up to 2 copies — compare against the write pointer 2 positions back |

### Fast/Slow Pointers

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 141 | Linked List Cycle | Easy | Same Floyd's algorithm as Section 4, applied to an actual linked list — you'll formalize this fully in `07_Linked_Lists` |
| 202 | Happy Number | Easy | The sequence of digit-square-sums either reaches 1 or cycles forever — treat "next number" as the implicit pointer, same as Section 4's array trick |
| 287 | Find the Duplicate Number | Medium | Section 4, directly |
| 142 | Linked List Cycle II | Medium | Find the cycle's ENTRANCE, not just whether a cycle exists — the two-phase technique from Section 4 |

### Merging / Sorted-Array Two Pointers

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 88 | Merge Sorted Array | Easy | Classic twist: merge from the BACK of the array (largest values first) to avoid overwriting unread elements in the first array |
| 977 | Squares of a Sorted Array | Easy | (listed above too — it's also a merge-style problem: two sorted sub-sequences of squares, one from each side of zero) |

### Three Pointers / Partitioning

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 75 | Sort Colors | Medium | Section 6, directly |
| 905 | Sort Array By Parity | Easy | Two-region version (even/odd) of the same partitioning idea, simpler than Dutch National Flag |

### Self-Check Before Moving to `02_Sliding_Window_and_Prefix_Sum`
If you can solve the Easy-tier problems above without re-reading this notebook, and can at least correctly IDENTIFY which pattern applies to each Medium/Hard problem (even if the implementation takes a couple of tries), you're ready for the next notebook. Sliding window is a close cousin of same-direction two pointers — the distinction will be much easier to hold onto with these fresh.
